In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

# transformers es una libreria de pytorch que permite cargar modelos de procesamiento de lenguaje natural preentrenados.
# Primero se implementa en pytorch y luego se convierte a tensorflow.

# transformers ya tiene implementadas las funciones de tokenizacion, embedding, etc.
# se puede cargar modelos de procesamiento de lenguaje natural preentrenados o entrenar modelos propios.



/Users/larenwell/Documents/personal/projects/ai-dl-tensorflow-specialization-fabricum-pucp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df_train = pd.read_csv("datasets-lab4/df_train.csv")
df_test = pd.read_csv("datasets-lab4/df_test.csv")

train_df, val_df = train_test_split(
    df_train,
    test_size=0.2,
    random_state=42,
    stratify=df_train["label"]
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [10]:
train_encodings = tokenizer(
    list(train_df["tweet"]),
    truncation=True,
    padding=True,
    max_length=128
)

val_encodings = tokenizer(
    list(val_df["tweet"]),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(df_test["tweet"]),
    truncation=True,
    padding=True,
    max_length=128
)

In [ ]:
# Lo que se mete a un modelo es x_train y y_train
# Para este caso lo estamos transformando en un dataset de tensorflow
# Además estamos mezclando los datos
# Dividimos en batches de 32 para que no se quede tan grande la memoria

train_dataset = tf.data.Dataset.from_tensor_slices((
    dict(train_encodings),
    train_df["label"].values
)).shuffle(1000).batch(32)

val_dataset = tf.data.Dataset.from_tensor_slices((
    dict(val_encodings),
    val_df["label"].values
)).batch(32)

In [12]:
model = TFAutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=3 # 3 clases
)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

In [14]:
optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)

loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

metrics = ["accuracy"]

model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
)

In [ ]:
model.summary()

In [ ]:
history = model.fit(
    train_dataset, # Es similar a aplicar: (train_pad_seqs, train_label_seqs)
    validation_data=val_dataset, # Es similar a aplicar: (val_pad_seqs, val_label_seqs)
    epochs=3
)

Epoch 1/3
 14/496 [..............................] - ETA: 54:52 - loss: 0.8210 - accuracy: 0.7433

In [ ]:
preds = model.predict(val_dataset)

y_pred = np.argmax(preds.logits, axis=1)
y_true = val_df["label"].values

print(classification_report(y_true, y_pred, digits=4))

In [ ]:
#model.save('modelo.h5')